# Camera Discovery Map UI — Colab Runner

This notebook installs and runs **camera-discovery-map-ui** in Google Colab. It can build the interactive map from the included sample fixtures or from a camera-discovery `artifacts.zip` / artifact directory.

Recommended flow:
1. Clone/install the repository.
2. Build and preview the sample map.
3. Upload a real camera-discovery `artifacts.zip`.
4. Build and preview the real review map.


## 1. Clone or locate the repository

If this notebook is already running from a checked-out repository at `/content/camera-discovery-map-ui`, the clone step is skipped. Otherwise, it clones the GitHub repository named below. Change `REPO_URL` if your repository lives under a different account or organization.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get(
    "CAMERA_DISCOVERY_MAP_UI_REPO",
    "https://github.com/dshipley71/camera-discovery-map-ui.git",
)
PROJECT_DIR = Path("/content/camera-discovery-map-ui")

if PROJECT_DIR.exists():
    print(f"Using existing repository: {PROJECT_DIR}")
else:
    print(f"Cloning {REPO_URL} -> {PROJECT_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")
print("Repository files:")
for item in sorted(PROJECT_DIR.iterdir()):
    print(" -", item.name)


## 2. Install the package and run smoke tests

The editable install lets the notebook use the local source tree. The test run is optional but useful to verify that the package and template contract are healthy in Colab.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

print("Installed camera-discovery-map-ui in editable mode.")
print("Running tests...")
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR)
if result.returncode != 0:
    print("Tests failed. You can continue to inspect the failure, but map generation may not be reliable.")


## 3. Build a map from the included fixtures

This verifies the application without requiring a real camera-discovery run. The sample fixture intentionally contains fake records such as `Fixture Camera 1`; it is only for smoke testing. If you see that record, you are looking at test data, not your camera-discovery output.

The generated output contains `camera_map_bundle.json`, `camera_map.geojson`, and `map.html`.


In [ ]:
from pathlib import Path
import json
import shutil

from camera_discovery_map_ui.agent import CameraDiscoveryMapAgent

SAMPLE_INPUT = PROJECT_DIR / "tests" / "fixtures"
SAMPLE_OUTPUT = Path("/content/camera-map-review-sample")

if SAMPLE_OUTPUT.exists():
    shutil.rmtree(SAMPLE_OUTPUT)

result = CameraDiscoveryMapAgent(input_path=SAMPLE_INPUT, output_dir=SAMPLE_OUTPUT).run()
print("Sample map artifacts written:")
print(" -", result.bundle_path)
print(" -", result.geojson_path)
print(" -", result.map_html_path)
print(json.dumps(result.summary, indent=2))


## 4. Display the sample map in Colab

The map is served over a local HTTP server because browser security rules often block `map.html` from loading local JSON files over `file://`.


In [ ]:
import functools
import http.server
import socketserver
import threading
from IPython.display import IFrame, display


def serve_map_directory(directory: Path, port: int = 8000, height: int = 900):
    # Serve a map output directory and display map.html in Colab or a local notebook.
    directory = Path(directory).resolve()
    assert (directory / "map.html").exists(), f"Missing map.html in {directory}"

    # Shut down a previous server started by this notebook cell, if present.
    global _camera_map_httpd
    try:
        _camera_map_httpd.shutdown()
        _camera_map_httpd.server_close()
    except Exception:
        pass

    handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=str(directory))

    class ReusableTCPServer(socketserver.TCPServer):
        allow_reuse_address = True

    _camera_map_httpd = ReusableTCPServer(("0.0.0.0", port), handler)
    thread = threading.Thread(target=_camera_map_httpd.serve_forever, daemon=True)
    thread.start()

    print(f"Serving {directory} on port {port}")
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, path="/map.html", height=height)
    except Exception:
        display(IFrame(src=f"http://localhost:{port}/map.html", width="100%", height=height))


serve_map_directory(SAMPLE_OUTPUT, port=8000, height=900)


## 5. Upload and build from a real camera-discovery artifact or GeoJSON

Upload one of the following:

- `artifacts.zip`,
- a ZIP/directory containing files such as `camera.geojson`, `untrusted_camera_candidates.geojson`, and `camera_candidates_table.csv`, or
- a single `.geojson` file.

If you open a `.geojson` directly in the browser, it will display raw JSON. To see the interactive UI, build the map and open the generated `map.html`.

If you already have the artifact in Colab, set `ARTIFACT_INPUT` manually and skip the upload cell.


In [ ]:
from pathlib import Path

ARTIFACT_INPUT = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded.keys()))
        ARTIFACT_INPUT = Path("/content") / first_name
        print(f"Using uploaded artifact: {ARTIFACT_INPUT}")
except Exception as exc:
    print("Colab upload helper is unavailable in this environment.")
    print("Set ARTIFACT_INPUT manually, for example:")
    print('ARTIFACT_INPUT = Path("/content/artifacts.zip")')

ARTIFACT_INPUT


## 6. Build and display the uploaded artifact map

Run this after `ARTIFACT_INPUT` points to an artifact ZIP or directory.


In [ ]:
from pathlib import Path
import json
import shutil

USER_OUTPUT = Path("/content/camera-map-review-user")

if ARTIFACT_INPUT is None:
    raise ValueError("ARTIFACT_INPUT is not set. Upload an artifact ZIP or set ARTIFACT_INPUT manually.")

if USER_OUTPUT.exists():
    shutil.rmtree(USER_OUTPUT)

result = CameraDiscoveryMapAgent(input_path=Path(ARTIFACT_INPUT), output_dir=USER_OUTPUT).run()
print("User map artifacts written:")
print(" -", result.bundle_path)
print(" -", result.geojson_path)
print(" -", result.map_html_path)
print(json.dumps(result.summary, indent=2))

serve_map_directory(USER_OUTPUT, port=8001, height=900)


## 7. Inspect the generated bundle

This cell prints summary counts and shows a small table of normalized rows. It helps verify how records were classified before you inspect them in the map UI.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

BUNDLE_PATH = USER_OUTPUT / "camera_map_bundle.json" if 'USER_OUTPUT' in globals() and (USER_OUTPUT / "camera_map_bundle.json").exists() else SAMPLE_OUTPUT / "camera_map_bundle.json"

bundle = json.loads(BUNDLE_PATH.read_text(encoding="utf-8"))
print("Bundle:", BUNDLE_PATH)
print(json.dumps(bundle.get("summary", {}), indent=2))

rows = bundle.get("rows", [])
if rows:
    columns = [
        "record_id", "label", "status", "validation_status", "trust_level",
        "scope_status", "camera_type", "media_type", "has_coordinates",
        "latitude", "longitude", "source_artifact",
    ]
    df = pd.DataFrame(rows)
    display(df[[c for c in columns if c in df.columns]].head(50))
else:
    print("No rows found in bundle.")


## 8. Download generated map output

This creates a ZIP containing the generated `map.html`, normalized bundle, and GeoJSON output.


In [ ]:
import shutil
from pathlib import Path

OUTPUT_TO_DOWNLOAD = USER_OUTPUT if 'USER_OUTPUT' in globals() and USER_OUTPUT.exists() else SAMPLE_OUTPUT
zip_base = Path("/content") / OUTPUT_TO_DOWNLOAD.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_TO_DOWNLOAD))
print(f"Created {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper unavailable. ZIP path:", zip_path)
